# 07 · Long-distance key retrieval sequence inspection

Implements v19.1's data recipe: randomly choose key A–D; insert exactly `distance` distractors drawn with replacement from x/y/z/n; append QUERY; classify the initial key. Inspect distances 50, 100, 200. This generator is a faithful rule-level reimplementation; neural accuracy requires the benchmark training run.

In [ ]:
from pathlib import Path
import json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project():
    candidates=[Path.cwd(), Path.cwd()/"project_inputs", Path.cwd().parent/"project_inputs", Path("/mnt/data/diina_reproducibility/project_inputs"), Path("/mnt/data")]
    for p in candidates:
        if (p/"raw_results.csv").exists(): return p
    raise FileNotFoundError("Place notebook with project_inputs/ or set cwd to extracted project directory")
PROJECT=find_project()
df=pd.read_csv(PROJECT/"raw_results.csv")
summary=json.loads((PROJECT/"summary.json").read_text())
print("Project files:",PROJECT,"| telemetry shape:",df.shape,"| columns:",list(df.columns))
assert len(df)==1500 and set(df.columns)=={"seed","step","loss","fast_usage","slow_usage","entropy"}

In [ ]:
src=(PROJECT/"DIINA_v19_1_LongDistance.py").read_text(); print("Source has distances 50/100/200:",all(str(x) in src for x in [50,100,200])); print("source generator excerpt:"); start=src.find("def create_dataset"); print(src[start:start+600])
rng=np.random.default_rng(42); keys=list("ABCD"); distractors=list("xyzn")
def generate(distance,rng=rng):
    key=rng.choice(keys); noise=rng.choice(distractors,size=distance,replace=True).tolist(); seq=[key,*noise,"QUERY"]
    return seq,key
for distance in [50,100,200]:
    seq,target=generate(distance); print({"distance":distance,"length":len(seq),"distractors":sum(t in distractors for t in seq),"target":target,"prefix":seq[:5],"suffix":seq[-3:]})
    assert len(seq)==distance+2 and seq[-1]=="QUERY" and seq[0]==target

In [ ]:
try:
    import ipywidgets as widgets
    @widgets.interact(distance=widgets.SelectionSlider(options=[50,100,200],value=100))
    def show_span(distance=100):
        seq,target=generate(distance); print(f"key index=0; distractor span=1..{distance}; query index={distance+1}; sequence length={len(seq)}; target={target}")
except ImportError: print("ipywidgets unavailable; call generate(distance)")

v19.0 additionally includes distance 500 and uses 800 training / 300 test samples. v19.1 uses 300 / 100 and 32-dimensional representations in the shown script. Each benchmark's Mamba class is a **GRU proxy**, and its DIINA is a **single-GRU plus sigmoid inhibition gate**.